In [5]:
pip install --upgrade pip

In [6]:
pip install tensorflow 

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import random
import sys
import os

In [2]:
df = pd.read_csv(r'C:\Users\alokk\Documents\GenAIWorkshop\LSTM\train.csv')
df

,title,text,subject,date
0,Greens say no support for Macron's EZ budget i...,BERLIN (Reuters) - None of the German parties ...,worldnews,"October 25, 2017"
1,Trump faces uphill battle to overcome court's ...,(Reuters) - U.S. President Donald Trump faces ...,politicsNews,"February 6, 2017"
2,Ukraine president denies hampering anti-corrup...,VILNIUS/KIEV (Reuters) - Ukrainian President P...,worldnews,"December 8, 2017"
3,U.S. defense chief: White House shakeup will n...,BRUSSELS (Reuters) - U.S. Defense Secretary Ji...,politicsNews,"February 14, 2017"
4,Irish government set to fall weeks before Brex...,DUBLIN (Reuters) - Ireland s minority governme...,worldnews,"November 24, 2017"
...,...,...,...,...
14986,British negotiators still working on Brexit de...,LONDON (Reuters) - British negotiators are sti...,worldnews,"November 29, 2017"
14987,Democratic voter-intimidation cases falter in ...,WASHINGTON (Reuters) - Democrats fell short in...,politicsNews,"November 7, 2016"
14988,Cyprus president to seek second five-year term...,NICOSIA (Reuters) - Cypriot President Nicos An...,worldnews,"October 14, 2017"
14989,Trump says strong Europe is important for U.S,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,"April 20, 2017"


In [3]:
text = " ".join(df['text'].dropna().astype(str)).lower()
print(f'Total characters in text: {len(text)}')

Total characters in text: 35695884


In [4]:
# Creating Vocabulary and Character Mappings
vocab = sorted(set(text))
print(f'Unique characters in text: {len(vocab)}')

Unique characters in text: 104


sorted(set(text)): Extracts unique characters and sorts them to form the vocabulary.
char2idx: Maps each character to a unique integer index.
idx2char: Maps integers back to characters and is used during text generation.
text_as_int: Converts the entire text into a sequence of integer indices.

In [5]:
char_to_idx = {char: idx for idx, char in enumerate(vocab)}
idx_to_char = {idx: char for idx, char in enumerate(vocab)}

text_as_int = np.array([char_to_idx[char] for char in text])
print(f'Text as integers: {text_as_int[:100]}')

Text as integers: [41 44 57 51 48 53  1  9 57 44 60 59 44 57 58 10  1 14  1 53 54 53 44  1
 54 45  1 59 47 44  1 46 44 57 52 40 53  1 55 40 57 59 48 44 58  1 48 53
 61 54 51 61 44 43  1 48 53  1 44 63 55 51 54 57 40 59 54 57 64  1 42 54
 40 51 48 59 48 54 53  1 59 40 51 50 58  1 58 60 55 55 54 57 59  1 45 57
 44 53 42 47]


In [6]:
# Pre-processing the Data
seq_length = 100
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(chunk):
    input_text = chunk[:-1]
    target_text = chunk[1:]
    return input_text, target_text

dataset = sequences.map(split_input_target)
BATCH_SIZE = 64
BUFFER_SIZE = 10000
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)
dataset

<_BatchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>

In [7]:
# building the model
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 1024

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_shape=[None]),
    tf.keras.layers.LSTM(rnn_units, return_sequences=True),
    tf.keras.layers.Dense(vocab_size)
])

c:\Users\alokk\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [8]:
def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)
model.compile(optimizer='adam', loss=loss)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 256)      │        26,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 1024)     │     5,246,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 104)      │       106,600 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,380,200 (20.52 MB)

 Trainable params: 5,380,200 (20.52 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#training the model
EPOCHS = 20
history = model.fit(dataset, epochs=EPOCHS)

In [ ]:
#generating text
def generate_text(model, start_string, num_generate=1000):
    input_eval = [char_to_idx[s] for s in start_string]
    input_eval = tf.expand_dims(input_eval, 0)
    text_generated = []
    temperature = 1.0

    model.reset_states()
    for i in range(num_generate):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0) / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1,0].numpy()
        input_eval = tf.expand_dims([predicted_id], 0)
        text_generated.append(idx_to_char[predicted_id])

    return start_string + ''.join(text_generated)
print(generate_text(model, start_string="Once upon a time, "))